# Angle A: train a ranking objective instead of Logloss

## Why

TS-AUC scores **within an online step, across series**: for step `s`, rank the series that
have broken above those that have not, then weight the step by `n_pos * n_neg`. Anything
constant within a step is invisible to it - a monotone per-step transform changes nothing.

Logloss does not know that. A large, easy part of the pointwise loss is the **time-trend**:
late steps are mostly post-break (pos-rate rises from ~0 to ~0.5), so predicting "later =
higher" cuts Logloss a lot while contributing **exactly zero** TS-AUC. Model capacity spent
on the trend is capacity not spent on separating series *within* a step.

This is a structural mismatch, not a missing feature - which is why it is worth trying after
[[honest-cv-harness]] showed feature increments are unmeasurably small.

## The setup

Group = online step. Each group then holds at most one row per series, so **equal-series
weighting comes for free** from the grouping - no `sample_weight` needed, unlike the
pointwise baseline which must supply it explicitly.

Note lambdarank's truncation level: it optimises NDCG@k, and the default k=30 only cares
about the top 30 of groups that hold thousands of series. TS-AUC cares about the **whole**
ordering, so truncation is tested as a variable, not left at default.

Configs, all on the folds already used in `cv_revalidate` (seed 0, 5 splits) so results are
directly comparable:
- **D_cat_v3** pointwise CatBoost - the shipped submit#3, re-run as the paired baseline
- **R_lgb_t30** LGBMRanker lambdarank, default truncation 30
- **R_lgb_t500** LGBMRanker lambdarank, truncation 500 (nearly the full list)
- **R_cat_yeti** CatBoost YetiRank

In [1]:
import os, sys, json, time
import numpy as np, pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier, CatBoostRanker, Pool

TOOLS = os.path.abspath('../tools')
sys.path.insert(0, TOOLS)
import cv_tools as CV

d = np.load(os.path.join(TOOLS, 'cv_folds_by_series', 'cv_cache_full.npz'))
X, y, sid, tim = d['X'], d['y'], d['sid'], d['time']
sampled, cols = d['is_sampled_step'], list(d['cols'])
index = pd.MultiIndex.from_arrays([sid, tim], names=['id', 'time'])
folds = CV.series_folds(sid, n_splits=5, seed=0)      # same seed as cv_revalidate
step = CV.steps_from_index(index)
print(f'{X.shape[0]:,} rows | {len(cols)} feats | sampled {sampled.sum():,}')

5,036,517 rows | 50 feats | sampled 328,996


In [2]:
# How much of the label is pure time-trend? If a step-only predictor already scores well on
# Logloss but 0.5 on TS-AUC, that is the capacity the ranking objective stops wasting.
prate = pd.Series(y[sampled]).groupby(step[sampled]).mean()
print('pos-rate by sampled step: first 5', prate.head().round(3).tolist(),
      '| last 5', prate.tail().round(3).tolist())
trend = pd.Series(step, index=index).map(prate).fillna(prate.mean())
print(f'TS-AUC of the pure time-trend predictor: {CV.ts_auc(trend, pd.Series(y, index=index)):.4f}')

pos-rate by sampled step: first 5 [0.003, 0.005, 0.007, 0.009, 0.011] | last 5 [0.643, 0.667, 0.462, 1.0, 0.714]


TS-AUC of the pure time-trend predictor: 0.5000


In [3]:
def grouped(split):
    """Sort a Split by online step and return contiguous group sizes.

    Both LightGBM and CatBoost require rows already blocked by group; feeding unsorted
    rows silently mis-assigns them rather than erroring."""
    o = np.argsort(split.step, kind='stable')
    s = split.step[o]
    sizes = np.diff(np.flatnonzero(np.r_[True, s[1:] != s[:-1], True]))
    assert sizes.sum() == len(o)
    return o, s, sizes


def lgb_rank_fp(trunc):
    def _f(train, val):
        o, _, sizes = grouped(train)
        m = lgb.LGBMRanker(objective='lambdarank', lambdarank_truncation_level=trunc,
                           label_gain=[0, 1], n_estimators=1500, learning_rate=0.02,
                           num_leaves=63, min_child_samples=300, subsample=0.8,
                           subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                           n_jobs=-1, verbosity=-1, random_state=0)
        # no sample_weight: one row per series per group already equalises series
        m.fit(train.X[o], train.y[o], group=sizes)
        return m.predict(val.X)
    return _f


def cat_yeti_fp(train, val):
    o, s, _ = grouped(train)
    m = CatBoostRanker(loss_function='YetiRank', iterations=1500, learning_rate=0.02,
                       depth=6, l2_leaf_reg=3.0, random_seed=0, verbose=0, thread_count=-1)
    m.fit(Pool(train.X[o], train.y[o], group_id=s))
    return m.predict(val.X)


CATP = dict(iterations=1500, learning_rate=0.02, depth=6, l2_leaf_reg=3.0,
            bootstrap_type='Bernoulli', subsample=0.8, loss_function='Logloss',
            random_seed=0, verbose=0, thread_count=-1)


def cat_point_fp(train, val):
    m = CatBoostClassifier(**CATP)
    m.fit(train.X, train.y, sample_weight=train.w)
    return m.predict_proba(val.X)[:, 1]

In [4]:
CONFIGS = [('D_cat_v3', cat_point_fp),
           ('R_lgb_t30', lgb_rank_fp(30)),
           ('R_lgb_t500', lgb_rank_fp(500)),
           ('R_cat_yeti', cat_yeti_fp)]

res = {}
for name, f in CONFIGS:
    t = time.time()
    print(f'\n=== {name} ===', flush=True)
    try:
        res[name] = CV.run_cv(X, y, sid, index, f, folds=folds,
                              train_row_mask=sampled, row_step=step)
        print(f'{res[name].summary(name)} | {time.time() - t:.0f}s', flush=True)
    except Exception as e:
        print(f'{name} FAILED after {time.time() - t:.0f}s: {type(e).__name__}: {e}', flush=True)


=== D_cat_v3 ===


  fold 0: train   263,351 rows / 8,000 series | val 1,012,996 rows / 2,000 series | TS-AUC 0.5849


  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5964


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5828


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6014


...

  fold 1: train   263,306 rows / 8,000 series | val   982,164 rows / 2,000 series | TS-AUC 0.5974


  fold 2: train   263,072 rows / 8,000 series | val 1,018,819 rows / 2,000 series | TS-AUC 0.5860


  fold 3: train   263,146 rows / 8,000 series | val 1,015,051 rows / 2,000 series | TS-AUC 0.6269


  fold 4: train   263,109 rows / 8,000 series | val 1,007,487 rows / 2,000 series | TS-AUC 0.6209


R_cat_yeti         mean 0.6049 +/- 0.0180 (sem 0.0081) | OOF 0.6046 | folds: 0.5931  0.5974  0.5860  0.6269  0.6209 | 23937s


In [5]:
print(CV.summarize(res))
print('\n--- ranking objective vs pointwise baseline ---')
for n in res:
    if n != 'D_cat_v3':
        print()
        print(CV.paired_compare(res[n], res['D_cat_v3'], n, 'D_cat_v3'))

R_cat_yeti         mean 0.6049 +/- 0.0180 (sem 0.0081) | OOF 0.6046 | folds: 0.5931  0.5974  0.5860  0.6269  0.6209
R_lgb_t30          mean 0.6018 +/- 0.0132 (sem 0.0059) | OOF 0.6015 | folds: 0.5908  0.5930  0.5956  0.6070  0.6226
D_cat_v3           mean 0.5977 +/- 0.0162 (sem 0.0073) | OOF 0.5968 | folds: 0.5849  0.5964  0.5828  0.6014  0.6232
R_lgb_t500         mean 0.5967 +/- 0.0160 (sem 0.0071) | OOF 0.5967 | folds: 0.5856  0.5856  0.5925  0.5955  0.6242

--- ranking objective vs pointwise baseline ---

R_lgb_t30 - D_cat_v3: +0.0041 +/- 0.0064 (sem 0.0028, t +1.43, wins 3/5) -> within noise
  per-fold deltas: +0.0059  -0.0034  +0.0129  +0.0057  -0.0007

R_lgb_t500 - D_cat_v3: -0.0010 +/- 0.0078 (sem 0.0035, t -0.30, wins 3/5) -> within noise
  per-fold deltas: +0.0007  -0.0108  +0.0097  -0.0059  +0.0010

R_cat_yeti - D_cat_v3: +0.0071 +/- 0.0110 (sem 0.0049, t +1.45, wins 4/5) -> within noise
  per-fold deltas: +0.0082  +0.0010  +0.0032  +0.0256  -0.0023


In [6]:
# Does the ranker decorrelate from pointwise? If so a blend may beat either, even when the
# ranker alone does not win.
if len(res) > 1:
    P = pd.DataFrame({n: CV.rank01(r.oof_pred.to_numpy()) for n, r in res.items()})
    print('OOF rank-correlation:'); print(P.corr(method='spearman').round(3))
    yt = pd.Series(y, index=index)
    for n in res:
        if n != 'D_cat_v3':
            b = pd.Series((P[n] + P['D_cat_v3']).to_numpy(), index=index)
            print(f'blend {n} + D_cat_v3: {CV.ts_auc(b, yt):.4f} '
                  f'(vs {res[n].oof_score:.4f} / {res["D_cat_v3"].oof_score:.4f} OOF)')

json.dump({n: dict(fold_scores=r.fold_scores.tolist(), mean=r.mean, std=r.std,
                   sem=r.sem, oof=r.oof_score) for n, r in res.items()},
          open('rank_results.json', 'w'), indent=2)
print('\nsaved rank_results.json')

OOF rank-correlation:


            D_cat_v3  R_lgb_t30  R_lgb_t500  R_cat_yeti
D_cat_v3       1.000      0.757       0.479       0.146
R_lgb_t30      0.757      1.000       0.771       0.484
R_lgb_t500     0.479      0.771       1.000       0.622
R_cat_yeti     0.146      0.484       0.622       1.000


blend R_lgb_t30 + D_cat_v3: 0.6037 (vs 0.6015 / 0.5968 OOF)


blend R_lgb_t500 + D_cat_v3: 0.6007 (vs 0.5967 / 0.5968 OOF)


blend R_cat_yeti + D_cat_v3: 0.6056 (vs 0.6046 / 0.5968 OOF)

saved rank_results.json
